# 🏆 Capstone Projects

## What You've Learned
Congratulations! You now know:

**LangChain:**
- Messages, Prompts, LCEL
- Runnables, Output Parsers
- Document Loaders, Text Splitters
- Embeddings, Vector Stores, Retrievers
- RAG pipelines
- Memory & Chat History
- Tools & Tool Calling
- Agents
- Callbacks, Streaming
- LangSmith, Structured Output
- Multi-modal, Guardrails
- Async, Testing, Deployment

**LangGraph:**
- State, Nodes, Edges
- Conditional Edges, Checkpoints
- Memory, Human-in-the-Loop
- Subgraphs
- Multi-Agent patterns (Supervisor, Reflection, Planner-Executor)
- Production patterns

## Capstone Projects
1. **PDF Research Assistant** — Upload PDFs, ask questions, cite sources
2. **Multi-Agent Code Generator** — Plan, write, review, test code
3. **Customer Support Agent** — With escalation and human-in-the-loop
4. **Financial Report Analyzer** — Multi-modal + structured output
5. **Autonomous Research System** — Multi-agent with web search

Each project is in a separate subfolder with full implementation.


In [ ]:
# ── Capstone 1: PDF Research Assistant ─────────────────────────────────
# Full production RAG application

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.documents import Document
import os

class RAGAssistantState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    vectorstore_ready: bool
    sources_used: list[str]

# Initialize components
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Global vectorstore (in production, use persistent storage)
vectorstore = None

def load_documents(pdf_paths: list[str]) -> FAISS:
    '''Load PDFs and create vector store.'''
    all_docs = []
    for path in pdf_paths:
        loader = PyPDFLoader(path)
        docs = loader.load()
        chunks = splitter.split_documents(docs)
        all_docs.extend(chunks)
    return FAISS.from_documents(all_docs, embeddings)

def answer_with_sources(state: RAGAssistantState) -> dict:
    '''Retrieve relevant docs and generate answer with citations.'''
    if not vectorstore:
        return {'messages': [AIMessage(content='Please load documents first.')]}

    last_human = next((m for m in reversed(state['messages']) if m.type == 'human'), None)
    if not last_human:
        return {}

    # Retrieve relevant chunks
    docs = vectorstore.similarity_search(last_human.content, k=3)
    context = '\n\n'.join([
        f'[Source: {doc.metadata.get("source", "unknown")}, Page {doc.metadata.get("page", "?")+ 1}]\n{doc.page_content}'
        for doc in docs
    ])
    sources = list(set(doc.metadata.get('source', 'unknown') for doc in docs))

    # Generate answer
    response = llm.invoke([
        SystemMessage(content=f'''Answer ONLY from the provided context.
Always cite the source. If not in context, say so.

Context:
{context}'''),
        last_human
    ])

    return {
        'messages': [response],
        'sources_used': sources
    }

# Build graph
graph = StateGraph(RAGAssistantState)
graph.add_node('answer', answer_with_sources)
graph.set_entry_point('answer')
graph.add_edge('answer', END)

checkpointer = MemorySaver()
rag_app = graph.compile(checkpointer=checkpointer)

print('PDF Research Assistant ready!')
print('Usage:')
print('  1. Call load_documents(["file1.pdf", "file2.pdf"])')
print('  2. Invoke rag_app with questions')
print('\nThis is a production-ready RAG assistant with:')
print('  ✓ Multi-PDF support')
print('  ✓ Source citations')
print('  ✓ Conversation memory')
print('  ✓ Persistent checkpoints')

In [ ]:
# ── Capstone 2: Multi-Agent Code Generator ──────────────────────────────
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

class CodeQuality(BaseModel):
    has_bugs: bool = Field(description='Does the code have bugs?')
    bugs_found: list[str] = Field(description='List of bugs found')
    quality_score: int = Field(description='Code quality 1-10', ge=1, le=10)
    approved: bool = Field(description='Is code ready for production?')

class CodeGenState(TypedDict):
    requirement: str
    plan: str
    code: str
    review_result: str
    iteration: int
    max_iterations: int

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)
reviewer_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(CodeQuality)

def planner_agent(state: CodeGenState) -> dict:
    plan = llm.invoke([
        SystemMessage(content='You are a software architect. Create a brief implementation plan.'),
        HumanMessage(content=f'Requirement: {state["requirement"]}')
    ]).content
    return {'plan': plan}

def coder_agent(state: CodeGenState) -> dict:
    prev_review = f'\nPrevious review issues: {state["review_result"]}' if state['review_result'] else ''
    code = llm.invoke([
        SystemMessage(content='You are a Python expert. Write clean, well-documented code.'),
        HumanMessage(content=f'Plan: {state["plan"]}\nRequirement: {state["requirement"]}{prev_review}\nWrite Python code:')
    ]).content
    return {'code': code, 'iteration': state['iteration'] + 1}

def reviewer_agent(state: CodeGenState) -> dict:
    review = reviewer_llm.invoke([
        SystemMessage(content='You are a senior code reviewer. Be strict.'),
        HumanMessage(content=f'Review this code for requirement: {state["requirement"]}\n\n{state["code"]}')
    ])
    bugs = ', '.join(review.bugs_found) if review.bugs_found else 'none'
    result = f'Score: {review.quality_score}/10. Bugs: {bugs}'
    print(f'Review iter {state["iteration"]}: {result}')
    return {'review_result': result, '__approved__': review.approved}

_last_approved = False

def review_and_route(state: CodeGenState) -> dict:
    global _last_approved
    review = reviewer_llm.invoke([
        SystemMessage(content='Review this code strictly.'),
        HumanMessage(content=f'Requirement: {state["requirement"]}\nCode:\n{state["code"]}')
    ])
    _last_approved = review.approved
    bugs = ', '.join(review.bugs_found) if review.bugs_found else 'none'
    print(f'Review {state["iteration"]}: Score {review.quality_score}/10, Approved: {review.approved}')
    return {'review_result': f'Score {review.quality_score}/10. Bugs: {bugs}'}

def should_revise(state: CodeGenState) -> Literal['coder', '__end__']:
    if state['iteration'] >= state['max_iterations'] or _last_approved:
        return '__end__'
    return 'coder'

graph = StateGraph(CodeGenState)
graph.add_node('planner', planner_agent)
graph.add_node('coder', coder_agent)
graph.add_node('reviewer', review_and_route)
graph.set_entry_point('planner')
graph.add_edge('planner', 'coder')
graph.add_edge('coder', 'reviewer')
graph.add_conditional_edges('reviewer', should_revise)
code_gen = graph.compile()

result = code_gen.invoke({
    'requirement': 'Write a Python function to find the nth Fibonacci number with memoization',
    'plan': '',
    'code': '',
    'review_result': '',
    'iteration': 0,
    'max_iterations': 3
})
print('\n=== FINAL CODE ===')
print(result['code'][:500], '...')

## 🎯 Interview Questions

1. **Walk me through the architecture of a production RAG application.**
2. **How would you scale a multi-agent system to handle 10,000 concurrent users?**
3. **What are the main failure modes of production LLM applications?**
4. **How do you evaluate the quality of a RAG system?**
5. **Design a customer support agent with human escalation. Describe the LangGraph graph.**
6. **How would you reduce LLM costs by 80% in production?**
7. **What is the difference between a chain, an agent, and a workflow?**
8. **How do you handle LLM hallucinations in a production RAG app?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Capstone Projects — Production AI Applications**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
